# Hierarchischer Ansatz: Logistische Regression

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** Regularisierte logistische Regression je Stufe im hierarchischen Ansatz testen (#99).

In [ ]:
%load_ext autoreload
%autoreload 2

from sklearn.linear_model import LogisticRegression

from awp2.config import LOGISTIC_REGRESSION_MAX_ITER, MODEL_DOCS_FIGURES_DIR
from awp2.data import load_split
from awp2.evaluation import compare_predictions
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import HierarchicalClassifier, summarize_hierarchical, summary_table
from awp2.models.tuned import refit_untracked, tuned_hierarchical_svm
from awp2.plots import plot_error_breakdown, plot_tuning_candidates, save_doc_figure
from awp2.preprocessing import PreprocessingConfig

In [ ]:
split = load_split()
split.X_train.shape

## Hypothese

Eine regularisierte multinomiale logistische Regression ist die einfachste lineare Referenz mit Klassengewichten. Nach LDA (#98) prüft sie, ob ein anderes lineares Modell die Kulturstufe besser trennt – oder ob die Nichtlinearität der RBF-SVM entscheidend ist.

Gleicher Ablauf wie bei der Referenz `hierarchical_svm` (#87): CV-Suche mit `tune()` auf dem
Trainingsteil, dann genau ein `run()`. Verglichen wird per CV mit der SVM-Referenz, danach
Fehlerzerlegung, Oracle-Kultur und McNemar gegen die SVM auf der Validierung.

In [ ]:
reference = tuned_hierarchical_svm()
svm_cv_score = reference.cv_score
preprocessing = PreprocessingConfig(use_spectral_standard_scale=True)
TOP_CANDIDATES = 15


def logistic_regression() -> LogisticRegression:
    return LogisticRegression(class_weight="balanced", max_iter=LOGISTIC_REGRESSION_MAX_ITER)


model = HierarchicalClassifier(logistic_regression(), logistic_regression())

strength = [0.1, 1.0, 10.0, 100.0]
param_grid = {"crop_estimator__C": strength, "stage_estimator__C": strength}

## CV-Suche

In [ ]:
tuned = tune(
    model,
    TuneConfig(
        name="hierarchical_logreg_search",
        approach="hierarchical",
        description="Logistische Regression je Stufe im hierarchischen Ansatz (#99).",
        preprocessing=preprocessing,
        balance_samples=False,
        param_grid=param_grid,
    ),
)
tuned.best_params, round(tuned.best_score, 4), round(tuned.best_score - svm_cv_score, 4)

In [ ]:
tuning_ax = plot_tuning_candidates(
    tuned.candidates.nsmallest(TOP_CANDIDATES, "rank"), color_by="crop_estimator__C"
)
tuning_ax.axvline(svm_cv_score, color="black", linestyle="--", linewidth=1)
tuning_ax.set_title(
    "Hierarchisch Logistische Regression: CV-Kandidaten (gestrichelt: SVM-Referenz)"
)
save_doc_figure(tuning_ax.figure, "hierarchical_logreg_tuning", MODEL_DOCS_FIGURES_DIR)

## Einmalige Bewertung

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="hierarchical_logreg",
        approach="hierarchical",
        description="Bester Logistische Regression aus hierarchical_logreg_search, "
        "einmal bewertet.",
        preprocessing=preprocessing,
        balance_samples=False,
        tuning_run=tuned.run_id,
    ),
)
result.metrics.model_dump(), result.run_id

## Vergleich mit der SVM-Referenz

Die SVM-Referenz (`tuned_hierarchical_svm()`) wird ohne MLflow nachtrainiert
(`refit_untracked`), um die Vorhersagen Zeile für Zeile zu vergleichen.

In [ ]:
svm_result = refit_untracked(reference, "hierarchical_svm_refit")
summaries = {
    "RBF-SVM (Referenz)": summarize_hierarchical(svm_result.pipeline, split.X_val, split.y_val),
    "Logistische Regression": summarize_hierarchical(result.pipeline, split.X_val, split.y_val),
}
breakdowns = {label: summary.breakdown for label, summary in summaries.items()}
(
    summary_table(summaries).round(3),
    compare_predictions(split.y_val, svm_result.y_pred, result.y_pred),
)

In [ ]:
breakdown_ax = plot_error_breakdown(breakdowns)
breakdown_ax.set_title("Fehlerzerlegung: Logistische Regression vs. RBF-SVM")
save_doc_figure(breakdown_ax.figure, "hierarchical_logreg_breakdown", MODEL_DOCS_FIGURES_DIR)

## Befund

- CV 0.747, Validierung 0.759 – deutlich unter der RBF-SVM (0.839 / 0.844), klar über LDA.
- 14 % falsche Kultur (SVM 9 %); mit wahrer Kultur 0.920. Wie bei LDA liegt der Abstand vor
  allem auf der Kulturstufe.
- Stadium `C=100` am Rasterrand: schwache Regularisierung wird bevorzugt.